# Trabajo Práctico AA1 - Clasificación

In [ ]:
%%capture
%pip install -r requirements.txt

In [ ]:
# manipulación, visualización y modelado.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# separación de datos, modelos y métricas.
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score, auc

# preprocesamiento.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# geolocalización y visualización de ubicaciones.
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from folium import Map, Marker, Icon
from sklearn.cluster import KMeans

## Carga dataset

In [ ]:
# Cargamos el dataset
df_original = pd.read_csv("content/weatherAUS_2026C1.csv")
#df_original = pd.read_csv("./content/weatherAUS_2026C1.csv")
df_original.head()

In [ ]:
#copiamos el dataset para mantener el original
df = df_original.copy()

In [ ]:
# revisamos las columnas
df.columns

In [ ]:
# cantidad de filas y columnas
df.shape

In [ ]:
# revisamos tipos de datos y valores no nulos
df.info()

### Descripción de variables

- Date: fecha de la observación.
- Location: ciudad o estación meteorológica donde se tomó la observación.
- MinTemp: temperatura mínima registrada durante el día.
- MaxTemp: temperatura máxima registrada durante el día.
- Rainfall: cantidad de lluvia registrada durante el día.
- Evaporation: evaporación registrada.
- Sunshine: cantidad de horas de sol.
- WindGustDir: dirección de la ráfaga de viento más fuerte.
- WindGustSpeed: velocidad de la ráfaga de viento más fuerte.
- WindDir9am: dirección del viento a las 9am.
- WindDir3pm: dirección del viento a las 3pm.
- WindSpeed9am: velocidad del viento a las 9am.
- WindSpeed3pm: velocidad del viento a las 3pm.
- Humidity9am: humedad a las 9am.
- Humidity3pm: humedad a las 3pm.
- Pressure9am: presión atmosférica a las 9am.
- Pressure3pm: presión atmosférica a las 3pm.
- Cloud9am: nubosidad a las 9am.
- Cloud3pm: nubosidad a las 3pm.
- Temp9am: temperatura a las 9am.
- Temp3pm: temperatura a las 3pm.
- RainToday: indica si llovió durante el día actual.
- RainTomorrow: variable objetivo; indica si llovió al día siguiente.
- RainfallTomorrow: Cantidad de lluvia al día siguiente

In [ ]:
df.isnull().sum()

Se detectan valores faltantes en varias columnas, por lo que será necesario tratarlo según el caso.

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

La columna "Unnamed: 0" parece ser un índice guardado en el archivo CSV, por lo que no representa una variable climática útil para la predicción. Por este motivo se elimina del análisis.

In [ ]:
df = df.drop(columns=["RainfallTomorrow"])

In [ ]:
# verificamos que hayan sido eliminadas
df.columns

La columna "RainfallTomorrow" no se utiliza como variable predictora porque contiene información asociada al día siguiente. Como el objetivo es predecir a las 23:59 hs si al día siguiente lloverá o no, esta información no estaría disponible al momento de realizar la predicción. Usarla produciría fuga de datos.

## EDA

### Locations

In [ ]:
def CamelCase(cadena: str) -> str:
  """
  Funcion para modificar el nombre de las variables porque sino tira error por como esta hecha.
  """
  resultado = ""
  for i, caracter in enumerate(cadena):
    if i > 0 and caracter.isupper() and cadena[i-1].islower():
      resultado += " "
    resultado += caracter
  return resultado

In [ ]:
ciudades = df['Location'].unique()
geolocator = Nominatim(user_agent="climate_region_clustering")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # evita baneos por rate limit
# Obtener coordenadas
coords = []
contador = 0
for ciudad in ciudades:
  location = geocode(f"{CamelCase(ciudad)}, Australia")
  if location:
    lat = location.latitude
    lon = location.longitude
    coords.append((ciudad, lat, lon))
    contador += 1
  else:
    print(f"No se encontró: {ciudad}")
    coords.append((ciudad, None, None))

df_coords = pd.DataFrame(coords, columns=["Location", "Latitude", "Longitude"])
print(f"Se cargaron correctamente {contador} ciudades.")

In [ ]:
map = Map(location=[-25.0, 133.0], zoom_start=4)
for _, row in df_coords.iterrows():
  Marker(location=[row['Latitude'], row['Longitude']], popup=row['Location'], icon=Icon()).add_to(map)
map

Ahora que tenemos las coordenadas y las ciudades ubicadas, las agrupamos en Regiones

In [ ]:
# Aplicar KMeans sobre latitud y longitud
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_coords["Region"] = kmeans.fit_predict(df_coords[["Latitude", "Longitude"]])

In [ ]:
colores = ['red', 'pink', 'green', 'purple', 'orange', 'darkred', 'darkgreen', 'darkblue', 'black', 'blue']

map = Map(location=[-25.0, 133.0], zoom_start=4)
for _, row in df_coords.iterrows():
  # Elegir el color según el cluster
  color = colores[row['Region'] % len(colores)]
  Marker(location=[row['Latitude'], row['Longitude']], popup=f"{row['Location']} (Cluster {row['Region']})", icon=Icon(color=color)).add_to(map)
map

In [ ]:
# Agregar Region al df original
df = df.merge(df_coords[['Location', 'Region']], on='Location', how='left')

In [ ]:
df

In [ ]:
df['Location'].value_counts().plot(kind='bar', figsize=(14, 6))
plt.title("Distribución de Location")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
df = df.drop(columns=['Location'])

In [ ]:
df['Region'].value_counts().sort_index().plot(kind='bar', figsize=(14, 6), color=colores)
plt.title("Distribución de Region")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

###

In [ ]:
# revisamos los valores de la variable objetivo
df["RainTomorrow"].value_counts(dropna=False)

In [ ]:
# Calculamos la proporción de cada clase
df["RainTomorrow"].value_counts(normalize=True, dropna=False)

In [ ]:
# eliminamos filas donde la variable objetivo no tiene valor
df = df.dropna(subset=["RainTomorrow"])
# verificamos
df["RainTomorrow"].value_counts(dropna=False)

In [ ]:
df_nuevo = df.copy()

In [ ]:
# separamos variables predictoras y variable objetivo.
X = df_nuevo.drop(columns=["RainTomorrow"])
y = df_nuevo["RainTomorrow"]
#y = y.map({"No": 0, "Yes": 1})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
df_train = pd.concat([X_train, y_train], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)

In [ ]:
type(y)

In [ ]:
type(y_train)

In [ ]:
print(len(X_train))
print(len(X_test))
print(len(y_train))
print(len(y_test))

In [ ]:
# Graficamos la distribución de la variable objetivo
sns.countplot(data=df_train,x="RainTomorrow", order=["Yes", "No"])
plt.title("Distribución de RainTomorrow")
plt.xlabel("Llueve mañana")
plt.ylabel("Cantidad de observaciones")
plt.show()

La variable objetivo del problema es "RainTomorrow", que indica si al día siguiente llovió o no. En distribución observamos que la clase "No" aparece con mucha mayor frecuencia que la clase "Yes". Por ende el dataset está desbalanceado, por este motivo, además de accuracy, será necesario observar métricas como precision, recall y F1-score, especialmente para la clase "Yes". Sino el modelo puede no aprender a detectar bien los dias que si llueve


In [ ]:
# eliminamos filas donde la variable objetivo no tiene valor
#df_train = df_train.dropna(subset=["RainTomorrow"])


In [ ]:
# verificamos
#df_train["RainTomorrow"].value_counts(dropna=False)

Las filas con valores faltantes en "RainTomorrow" se eliminan porque esta columna es la variable objetivo. Si no se conoce si al día siguiente llovió o no, esa observación no puede utilizarse para entrenar ni evaluar un modelo supervisado.

In [ ]:
df_train.dtypes

In [ ]:
# calculamos cantidad y porcentaje de valores faltantes por columna
faltantes = pd.DataFrame({
    "cantidad_faltantes": df_train.isna().sum(),
    "porcentaje_faltantes": df_train.isna().mean() * 100
})

faltantes = faltantes.sort_values(by="porcentaje_faltantes", ascending=False)

faltantes

Observamos que algunas variables presentan un porcentaje alto de valores faltantes, por este motivo no se eliminan filas completas con valores faltantes, ya que eso implicaría perder una gran cantidad de observaciones. Se decide conservar inicialmente las variables y realizar la imputación más adelante después de separar los datos en entrenamiento y prueba.


In [ ]:
# identificamos variables numéricas y categóricas
columnas_numericas = df_train.select_dtypes(include=["int64", "float64"]).columns
columnas_categoricas = df_train.select_dtypes(include=["object","int32"]).columns

print("Variables numéricas:")
print(columnas_numericas)

print("\nVariables categóricas:")
print(columnas_categoricas)

In [ ]:
# obtenemos medidas descriptivas de las variables numéricas
df_train[columnas_numericas].describe().T

Calculamos las medidas para las variables numéricas para detectar diferencias de escala entre variables

In [ ]:
# obtenemos un resumen de las variables categóricas.
df_train[columnas_categoricas].describe().T

En este caso interesa observar cuántas categorías distintas tiene cada variable y cuál es el valor más frecuente

### Histogramas

In [ ]:
# graficamos histogramas para observar la distribución de las variables numéricas.
df_train[columnas_numericas].hist(figsize=(15, 12), bins=30)
plt.suptitle("Distribución de variables numéricas")
plt.tight_layout()
plt.show()

Graficamos histogramas de las variables numéricas para observar su distribución. Observamos que las variables no se encuentran en la misma escala, por ejemplo, temperatura, presión, humedad, velocidad del viento y lluvia tienen unidades y rangos distintos. Por este motivo, más adelante será necesario aplicar escalado

### Boxplots

In [ ]:
# Graficamos boxplots individuales para todas las variables numéricas.
for columna in columnas_numericas:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_train, x=columna)
    plt.title(f"Boxplot de {columna}")
    plt.xlabel(columna)
    plt.show()

In [ ]:
# calculamos outliers por variable numérica usando el criterio del rango intercuartílico
resumen_outliers = []

for columna in columnas_numericas:
    q1 = df_train[columna].quantile(0.25)
    q3 = df_train[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    cantidad_outliers = ((df_train[columna] < limite_inferior) | (df_train[columna] > limite_superior)).sum()

    porcentaje_outliers = cantidad_outliers / df_train[columna].notna().sum() * 100

    resumen_outliers.append({
        "variable": columna,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "cantidad_outliers": cantidad_outliers,
        "porcentaje_outliers": porcentaje_outliers
    })

resumen_outliers = pd.DataFrame(resumen_outliers)
resumen_outliers.sort_values(by="porcentaje_outliers", ascending=False)

In [ ]:
# verificamos si existen valores negativos en variables que no deberían ser negativas
variables_no_negativas = ["Rainfall", "Evaporation", "Sunshine","WindGustSpeed", "WindSpeed9am", "WindSpeed3pm","Humidity9am", "Humidity3pm","Pressure9am", "Pressure3pm", "Cloud9am", "Cloud3pm"]

valores_negativos = {}

for columna in variables_no_negativas:
    cantidad_negativos = (df_train[columna] < 0).sum()
    valores_negativos[columna] = cantidad_negativos

pd.Series(valores_negativos).sort_values(ascending=False)

In [ ]:
# observamos los valores más frecuentes de Rainfall
df_train["Rainfall"].value_counts().head(20)

Analizamos posibles valores extremos utilizando el criterio del rango intercuartílico (IQR). Este criterio marca como posibles outliers los valores menores a Q1 - 1.5 * IQR o mayores a Q3 + 1.5 * IQR. Observamos también que una gran parte de las observaciones tiene lluvia igual o cercana a 0 mm. Esto hace que el rango intercuartílico sea bajo y que valores de lluvia mayores a 2 mm sean marcados como outliers. Estos valores no necesariamente representan errores, sino que pueden ser días con mayor cantidad de lluvia.

### Matriz de correlación

In [ ]:
# calculamos la matriz de correlación entre variables numéricas
correlacion = df_train[columnas_numericas].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlacion, cmap="coolwarm", center=0, annot=True)
plt.title("Matriz de correlación entre variables numéricas")
plt.xticks(rotation=45)
plt.show()

Se identifican correlaciones positivas fuertes entre variables relacionadas con la temperatura, como "MaxTemp" y "Temp3pm", y entre variables de presión, como "Pressure9am" y "Pressure3pm", también se observan correlaciones negativas entre "Sunshine" y variables como "Cloud9am", "Cloud3pm" y "Humidity3pm", lo cual es coherente con el clima, a mayor nubosidad o humedad o menor cantidad de horas de sol.
Esta matriz se utiliza como análisis exploratorio. No se eliminan variables automáticamente por correlación en esta etapa

In [ ]:
df_train[columnas_categoricas].nunique()

In [ ]:
# convertimos Date a formato fecha y extraemos el mes como variable numérica
df_train["Date"] = pd.to_datetime(df_train["Date"])
df_train["Month"] = df_train["Date"].dt.month

df_train = df_train.drop(columns=["Date"])

df_test["Date"] = pd.to_datetime(df_test["Date"])
df_test["Month"] = df_test["Date"].dt.month
# eliminamos la columna Date original para evitar codificar fechas completas

df_test = df_test.drop(columns=["Date"])

## 

Cambiamos la variable mes por Season para bajar las opciones de la variable de 12 a 4 y bajar la dimensionalidad del modelo

In [ ]:
def asignar_estacion(mes):
    if mes in [12, 1, 2]:
        return 'Summer'
    elif mes in [3, 4, 5]:
        return 'Autumn'
    elif mes in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

df_train['Season'] = df_train['Month'].apply(asignar_estacion)
# Eliminamos la variable original
df_train.drop('Month', axis=1, inplace = True)
df_test['Season'] = df_test['Month'].apply(asignar_estacion)
# Eliminamos la variable original
df_test.drop('Month', axis=1, inplace = True)

In [ ]:
X_train = df_train.drop(columns=["RainTomorrow"])
X_test = df_test.drop(columns=["RainTomorrow"])

y_train = df_train["RainTomorrow"]
y_test = df_test["RainTomorrow"]

Vemos que Location tiene demasiadas categorias en comparacion con las demas porque son ciudades, podemos reducirla agrupando por regiones mas grandes en vez de por ciudad

La columna "Date" se convierte a formato fecha y se utiliza para extraer el mes de la observación, lo que perermite conservar información  sobre estaciones, ya que la probabilidad de lluvia puede variar según la época del año. Luego se elimina la fecha completa, porque codificar cada día como una categoría distinta generaría muchas columnas y no aportaría una representación simple para el modelo.

In [ ]:
#X = df_modelo.drop(columns=["RainTomorrow"])
#y = df_modelo["RainTomorrow"]

In [ ]:
#y = y.map({"No": 0, "Yes": 1})

In [ ]:
# codificamos la variable objetivo: No = 0, Yes = 1.
y_train = y_train.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})

In [ ]:
y = pd.concat([y_train, y_test])

La variable objetivo se codifica como variable binaria donde "No" se representa con 0 y "Yes" con 1 para que la clase positiva corresponmda a los dias en los que si llueve

In [ ]:
# ahora identificamos variables numéricas y categóricas dentro de X
columnas_numericas = X_train.select_dtypes(include=["int64", "float64"]).columns
columnas_categoricas = X_train.select_dtypes(include=["object"]).columns

print("Variables numéricas:")
print(columnas_numericas)

print("\nVariables categóricas:")
print(columnas_categoricas)

Se vuelven a identificar las columnas numéricas y categóricas únicamente dentro de "X" para no incluir la variable objetivo dentro de las predictoras

In [ ]:
#X_train, X_test, y_train, y_test = df_train.drop(columns=["RainTomorrow"]), df_test.drop(columns=["RainTomorrow"]), df_train["RainTomorrow"], df_test["RainTomorrow"]

In [ ]:
# dividimos los datos en entrenamiento y prueba manteniendo la proporción de clases
#X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
# verificamos que train y test mantengan proporciones similares de clases
print("Proporción total:")
print(y.value_counts(normalize=True))

print("\nProporción en entrenamiento:")
print(y_train.value_counts(normalize=True))

print("\nProporción en prueba:")
print(y_test.value_counts(normalize=True))

Se divide el dataset en conjuntos de entrenamiento y prueba. Usamos un 20% de los datos para prueba y se mantiene la proporción de clases mediante "stratify=y", ya que la variable objetivo está desbalanceada

In [ ]:
# definimos el preprocesamiento para variables numéricas
preprocesamiento_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# definimos el preprocesamiento para variables categóricas
preprocesamiento_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# combinamos ambos preprocesamientos
preprocesamiento = ColumnTransformer(transformers=[
    ("num", preprocesamiento_numerico, columnas_numericas),
    ("cat", preprocesamiento_categorico, columnas_categoricas)
])

Separamos el preprocesamiento según tipo de variable. Para las numéricas usamos imputación con mediana y escalado estándar. Para las categóricas usamos imputación con la categoría más frecuente y codificación one-hot. Todo esto se coloca dentro de un Pipeline y un ColumnTransformer para que se ajuste solo con el conjunto de entrenamiento y así evitar fuga de datos.

## Ítem 2: Regresión logística

In [ ]:
# creamos un Pipeline que combina el preprocesamiento y la regresión logística
modelo_logistico = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
#Entrenamiento que estoy pasadno al modelo
X_train 

In [ ]:
# entrenamos el modelo utilizando el conjunto de entrenamiento
modelo_logistico.fit(X_train, y_train)

In [ ]:
# obtenemos las predicciones de clase sobre el conjunto de prueba
y_pred_logistico = modelo_logistico.predict(X_test)

In [ ]:
# obtenemos las probabilidades de pertenecer a la clase positiva
y_proba_logistico = modelo_logistico.predict_proba(X_test)[:, 1]

In [ ]:
# observamos las primeras predicciones de clase
y_pred_logistico[:10]

In [ ]:
# observamos las primeras probabilidades estimadas de lluvia
y_proba_logistico[:10]

In [ ]:
y_test[:10]

In [ ]:
y_pred_logistico[:10]

In [ ]:
# calculamos métricas principales para evaluar el modelo de regresión logística
accuracy_logistico = accuracy_score(y_test, y_pred_logistico)
precision_logistico = precision_score(y_test, y_pred_logistico)
recall_logistico = recall_score(y_test, y_pred_logistico)
f1_logistico = f1_score(y_test, y_pred_logistico)
auc_logistico = roc_auc_score(y_test, y_proba_logistico)

print("Accuracy:", accuracy_logistico)
print("Precision:", precision_logistico)
print("Recall:", recall_logistico)
print("F1-score:", f1_logistico)
print("ROC-AUC:", auc_logistico)

Se calculan distintas métricas porque el dataset está desbalanceado. En este caso, accuracy por si sola puede ser insuficiente, ya que un modelo podría acertar muchos casos de la clase mayoritaria (No) y aun así detectar mal los casos de lluvia (Yes). Por este motivo se analizan también precision, recall, F1-score y ROC-AUC. La clase positiva corresponde a "RainTomorrow = Yes", es decir, los días en los que sí llueve al día siguiente.

Con el umbral por defecto de 0.5, si la probabilidad es mayor o igual a 0.5 el modelo predice lluvia, si es menor, predice que no llueve. Con este umbral el modelo obtuvo una accuracy de 0.847, pero el recall fue de 0.508. Esto significa que aunque acierta muchos casos en general, detecta aproximadamente la mitad de los días en los que realmente llueve.

### Matriz de confusión

In [ ]:
# graficamos la matriz de confusión para la regresión logística
cm_logistico = confusion_matrix(y_test, y_pred_logistico)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_logistico,display_labels=["No", "Yes"])

disp.plot()
plt.title("Matriz de confusión - Regresión logística")
plt.show()

In [ ]:
# extraemos los valores de la matriz de confusión
vn, fp, fn, vp = cm_logistico.ravel()

print("Verdaderos negativos:", vn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdaderos positivos:", vp)

La matriz de confusión muestra que el modelo clasifica correctamente una gran cantidad de días sin lluvia, con 20862 verdaderos negativos. También detecta 3239 días lluviosos correctamente.

Sin embargo, se observan 3135 falsos negativos, es decir días en los que realmente llovió pero el modelo predijo que no llovería. Este valor es importante porque indica que el modelo todavía pierde una cantidad considerable de casos de lluvia.

También aparecen 1195 falsos positivos que corresponden a días en los que el modelo predijo lluvia pero finalmente no llovió. En este problema, los falsos positivos representan falsas alarmas, mientras que los falsos negativos representan lluvias no detectadas.

Esta matriz confirma lo que observamos en las métricas, el modelo tiene una buena accuracy general, pero el recall de la clase "Yes" es moderado, ya que no logra detectar todos los días lluviosos.

### Curva ROC y AUC

In [ ]:
# Calculamos la curva ROC para la regresión logística.
fpr_logistico, tpr_logistico, thresholds_logistico = roc_curve(y_test,y_proba_logistico)

plt.figure(figsize=(8, 6))
plt.plot(fpr_logistico,tpr_logistico,label=f"Regresión logística (AUC = {auc_logistico:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Clasificador aleatorio")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Regresión logística")
plt.legend()
plt.show()

 El ROC-AUC fue de 0.870 lo cual indica que las probabilidades del modelo separan bastante bien los casos de lluvia y no lluvia, por eso el problema no necesariamente está en que el modelo no sirva, sino en que el umbral 0.5 puede no ser el más conveniente si queremos detectar más días de lluvia. Bajar el umbral podría aumentar el recall para detectar más lluvias reales pero la desventaja es que también podrían aumentar los falsos positivos

### Análisis del umbral de decisión

In [ ]:
# buscamos un umbral posible maximizando la diferencia entre TPR y FPR
indice_mejor_umbral = np.argmax(tpr_logistico - fpr_logistico)
mejor_umbral = thresholds_logistico[indice_mejor_umbral]

print("Mejor umbral sugerido:", mejor_umbral)
print("TPR:", tpr_logistico[indice_mejor_umbral])
print("FPR:", fpr_logistico[indice_mejor_umbral])

In [ ]:
# generamos nuevas predicciones usando el umbral sugerido
y_pred_logistico_umbral = (y_proba_logistico >= mejor_umbral).astype(int)

print(classification_report(y_test, y_pred_logistico_umbral))

Con este nuevo umbral, el recall de la clase "Yes" aumenta de aproximadamente 0.51 a 0.80. Esto significa que el modelo detecta una mayor proporción de los días en los que realmente llueve. La precision de la clase "Yes" es 0.50 y la accuracy general disminuye a 0.78, esto pasa porque al bajar el umbral el modelo clasifica más casos como lluvia, lo que reduce los falsos negativos pero aumenta los falsos positivos.

Si se prioriza no perder lluvias reales conviene un umbral menor y si se prioriza evitar falsas alarmas conviene un umbral más alto.

## Ítem 3: implementación del modelo base

In [ ]:
# implementamos modelo base con "DummyClassifier", predice siempre el que tiene mas frecuencia
modelo_base = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("clasificador", DummyClassifier(strategy="most_frequent", random_state=42))
    ])

modelo_base.fit(X_train, y_train)
y_pred_base = modelo_base.predict(X_test)

In [ ]:
# calculamos métricas principales para evaluar el modelo base basado en frecuencia 
accuracy_base = accuracy_score(y_test, y_pred_base)
precision_base = precision_score(y_test, y_pred_base, zero_division=0)
recall_base = recall_score(y_test, y_pred_base, zero_division=0)
f1_base = f1_score(y_test, y_pred_base, zero_division=0)
auc_base = roc_auc_score(y_test, y_pred_base)

print("Accuracy:", accuracy_base)
print("Precision:", precision_base)
print("Recall:", recall_base)
print("F1-score:", f1_base)
print("ROC-AUC:", auc_base)

In [ ]:
# graficamos la matriz de confusión para la regresión logística
cm_base = confusion_matrix(y_test, y_pred_base)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_base,display_labels=["No", "Yes"])

disp.plot()
plt.title("Matriz de confusión - Regresión logística")
plt.show()

In [ ]:
# extraemos los valores de la matriz de confusión 
vn, fp, fn, vp = cm_base.ravel()

print("Verdaderos negativos:", vn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdaderos positivos:", vp)

La matriz de confusión del modelo base muestra que el clasificador predice siempre la clase mayoritaria, es decir, No. Por eso clasifica correctamente los días sin lluvia, pero no detecta ningún caso de lluvia.
Esto confirma que la accuracy no alcanza para evaluar este problema, ya que un modelo puede obtener un valor aparentemente aceptable simplemente prediciendo siempre la clase más frecuente.

Modelo Base, si llovio hoy -Z llueve mañana

In [ ]:

df_rain_train = X_train[["RainToday"]].copy()
df_rain_train["RainTomorrow"] = y_train.map({0: "No", 1: "Yes"})

# eliminamos filas donde RainToday sea nulo para observar la relación de forma directa
df_rain_train = df_rain_train.dropna(subset=["RainToday"])

tabla_rain_pct = pd.crosstab(
    df_rain_train["RainToday"],
    df_rain_train["RainTomorrow"],
    normalize="index"
) * 100

tabla_rain_pct = tabla_rain_pct.reindex(columns=["No", "Yes"])
tabla_rain_pct.round(2)

In [ ]:
tabla_rain_pct.plot(kind="bar", stacked=True, figsize=(7, 5))

plt.title("Relación entre RainToday y RainTomorrow")
plt.xlabel("¿Llovió hoy?")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.legend(title="¿Llueve mañana?")
plt.show()

En este caso vemos que cuando no llueve en el dia es muy probable que al dia siguiente tampoco llueva, pero para los dias que si llovió aunque se acerca un poco en su mayoria tampcoo llueve al dia siguiente deberia ser una mejor prediccion que simplemente predecir aue no llueve como en el anterior

In [ ]:
# implementamos un modelo base basado en una regla simple:
# si RainToday = Yes, predecimos RainTomorrow = Yes; si RainToday = No, predecimos RainTomorrow = No

moda_raintoday_train = X_train["RainToday"].mode()[0]

y_pred_base_raintoday = (
    X_test["RainToday"]
    .fillna(moda_raintoday_train)
    .map({"No": 0, "Yes": 1})
    .astype(int)
)

In [ ]:
# calculamos métricas principales para evaluar el modelo base basado en RainToday
accuracy_base_raintoday = accuracy_score(y_test, y_pred_base_raintoday)
precision_base_raintoday = precision_score(y_test, y_pred_base_raintoday, zero_division=0)
recall_base_raintoday = recall_score(y_test, y_pred_base_raintoday, zero_division=0)
f1_base_raintoday = f1_score(y_test, y_pred_base_raintoday, zero_division=0)
auc_base_raintoday = roc_auc_score(y_test, y_pred_base_raintoday)

print("Accuracy:", accuracy_base_raintoday)
print("Precision:", precision_base_raintoday)
print("Recall:", recall_base_raintoday)
print("F1-score:", f1_base_raintoday)
print("ROC-AUC:", auc_base_raintoday)

El baseline basado en la variable raintoday tiene un poco menor Accuracy que la que predice "No" pero aumenta precision y recall ya que predice positivo para algunos casos.

In [ ]:
# graficamos la matriz de confusión para la regresión logística
cm_rain_today = confusion_matrix(y_test, y_pred_base_raintoday)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_rain_today,display_labels=["No", "Yes"])

disp.plot()
plt.title("Matriz de confusión - Regresión logística")
plt.show()

In [ ]:
# extraemos los valores de la matriz de confusión
vn, fp, fn, vp = cm_rain_today.ravel()

print("Verdaderos negativos:", vn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdaderos positivos:", vp)

In [ ]:
# copiamos las variables numéricas
df_corr = df_train[columnas_numericas].copy()

# agregamos el target codificado como variable binaria
df_corr["RainTomorrow"] = y_train

# calculamos la matriz de correlación incluyendo el target
correlacion2 = df_corr.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlacion2, cmap="coolwarm", center=0, annot=True)
plt.title("Matriz de correlación entre variables numéricas y RainTomorrow")
plt.xticks(rotation=45)
plt.show()

Elegimos usar la variable Sunshine para la regresion logistica de 1 variable

In [ ]:
X_train_variable_unica = X_train[['Sunshine']]
X_test_unica = X_test[['Sunshine']]

# creamos un Pipeline que combina el preprocesamiento y la regresión logística
modelo_logistico_una_var = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento_numerico),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

# entrenamos el modelo utilizando el conjunto de entrenamiento
modelo_logistico_una_var.fit(X_train_variable_unica, y_train)

# obtenemos las predicciones de clase sobre el conjunto de prueba
y_pred_una_var = modelo_logistico_una_var.predict(X_test_unica)

# obtenemos las probabilidades de pertenecer a la clase positiva
y_proba_logistico_una_var = modelo_logistico_una_var.predict_proba(X_test_unica)[:, 1]


In [ ]:
# calculamos métricas principales para evaluar el modelo de regresión logística
accuracy_logistico = accuracy_score(y_test, y_pred_una_var)
precision_logistico = precision_score(y_test, y_pred_una_var)
recall_logistico = recall_score(y_test, y_pred_una_var)
f1_logistico = f1_score(y_test, y_pred_una_var)
auc_logistico = roc_auc_score(y_test, y_proba_logistico_una_var)

print("Accuracy:", accuracy_logistico)
print("Precision:", precision_logistico)
print("Recall:", recall_logistico)
print("F1-score:", f1_logistico)
print("ROC-AUC:", auc_logistico)

In [ ]:
# graficamos la matriz de confusión para la regresión logística
cm_una_var = confusion_matrix(y_test, y_pred_una_var)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_una_var,display_labels=["No", "Yes"])

disp.plot()
plt.title("Matriz de confusión - Regresión logística")
plt.show()

In [ ]:
# extraemos los valores de la matriz de confusión
vn, fp, fn, vp = cm_una_var.ravel()

print("Verdaderos negativos:", vn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdaderos positivos:", vp)

La regresión logística con todas las variables supera a los modelos base porque, además de mantener una accuracy mayor, logra identificar una parte de los días con lluvia. Esto se observa especialmente en el recall y el F1-score de la clase positiva.

Por lo tanto, la regresión logística aporta valor respecto de una estrategia ingenua basada solamente en predecir siempre la clase mayoritaria, basada en el valor de una variable o la regresion logisticca de una osla variable.

## Item 4: Implementamos optimización de hiperparámetros

Optimizaremos:
* C porque controla la regularización, entonces probamos un rango amplio para  encontrar el equilibrio que mejor generalice y evitar overfitting
* class_weight, porque permite ajustar el peso de cada clase durante el entrenamiento e incluímos balanced para que el modelo no ignore la clase minoritaria (lluvia), por estar desbalanceado el dataset
  
  

In [ ]:
# Grid search

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_params = {
    'clasificador__C': [0.01, 0.1, 1, 10, 100, 1000, 10000], # 7 valores
    'clasificador__class_weight': [None, 'balanced'] # 2 valores
}

# 7*2 = 14 combinaciones de hiperparámetros
# utilizamos 5 splits, entonces:
# nro. de entrenamientos = 5 * 14 = 70

modelo_grid = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("clasificador", LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

grid_search = GridSearchCV(
    modelo_grid, grid_params, cv=cv, scoring='f1', n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_grid_model = grid_search.best_estimator_
grid_search_f1 = f1_score(y_test, best_grid_model.predict(X_test))
accuracy_grid = accuracy_score(y_test, best_grid_model.predict(X_test))
precision_grid = precision_score(y_test, best_grid_model.predict(X_test))
recall_grid = recall_score(y_test, best_grid_model.predict(X_test))

Utilizamos validación cruzada "k-folds" para obtener una estimación más robusta de F1-score, promediando los scores sobre distintas partes (folds) del dataset. Aplicamos StratifiedKFold para que las clases queden proporcionadas en cada fold, así evitamos cualquier sesgo en los datos 

Elegimos grid search por sobre las otras dos opciones porque el espacio de parámetros es muy pequeño (14 combinaciones) y en estos casos grid search es más eficiente y consume menos recursos para realizar la búsqueda que Optuna o Random Search

In [ ]:
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor score:", grid_search.best_score_)
print("Accuracy:", accuracy_grid)
print("Precision:", precision_grid)
print("Recall:", recall_grid)
print("F1-score:", grid_search_f1)

Optimizando los hiperparámetros del modelo con grid search, logramos un F1-score de 0.62 en test, superando al dummy (F1 = 0) y al modelo base asumiendo que si llovió hoy, llueve mañana (F1 = 0.45)

La mejora respecto al modelo de regresión logística del item 2 (F1 = 0.59) es muy poca, lo que sugiere que ya alcanzamos el límite del modelo de regresión logística con este dataset

In [ ]:
best_grid_model

In [ ]:
preprocesamiento = best_grid_model.named_steps['preprocesamiento'] # ColumnTransformer
clasificador = best_grid_model.named_steps['clasificador'] # LogisticRegression

Desempaquetamos el pipeline, extrayendo preprocesador y clasificador

In [ ]:
X_train_scaled = preprocesamiento.transform(X_train)
X_test_scaled = preprocesamiento.transform(X_test)

Transformamos los datos para que Shap los reciba preprocesados como el clasificador del pipeline (pasan de df a array)

In [ ]:
OHE = preprocesamiento.named_transformers_["cat"].named_steps["encoder"]
columnas_categoricas_ohe = OHE.get_feature_names_out(columnas_categoricas)
feature_names = list(columnas_numericas) + list(columnas_categoricas_ohe)

Obtenemos el nombre de todas las columnas, numéricas + categóricas expandidas por OneHotEncoder

## Item 5: Explicabilidad de modelos mediante SHAP

In [ ]:
# Crea un objeto explainer SHAP
explainer = shap.LinearExplainer(clasificador, X_train_scaled, feature_names=feature_names)

# Calcula los valores SHAP para un conjunto de ejemplos de prueba
shap_values = explainer.shap_values(X_test_scaled)

In [ ]:
print(explainer.expected_value)

Como el dataset está desbalanceado, el modelo predice (sin tener en cuenta las features) que hay más probabilidad que el día siguiente a la medición no llueva

In [ ]:
shap_values.shape

In [ ]:
shap_values[0]

Gráficas a nivel local

In [ ]:
index=0
shap.force_plot(explainer.expected_value, shap_values[index], X_test_scaled[index],
feature_names=feature_names, matplotlib=True, figsize=(18, 4),
text_rotation=45)

Vemos que para la primer medición del set de test, las features empujan la predicción desde el valor base (aprox. -0.6), hasta -1.86. Es decir, las features que indican que no va a llover mañana tienen mayor peso que las features que indican que lloverá. Por lo tanto, es poco probable que llueva el día siguiente a la medición de prueba.

El valor que obtuvimos es logit = -1.86 (logaritmo de las chances). Esta transformación convierte la probabilidad (acotada entre 0 y 1) en una variable no
acotada, lo cual permite modelarla mediante una combinación lineal de las variables.

Podemos pasarlo a probabilidad con la fórmula p(x) = 1 / (1 + e^-(logit)).  En este caso, p(x) = 0.134 (13.4% de probabilidad de lluvia el día posterior a la medición)

In [ ]:
explanation = shap.Explanation(values=shap_values[index], base_values=explainer.expected_value, feature_names=feature_names)

In [ ]:
shap.plots.waterfall(explanation, max_display=5)

Partimos de E[f(x)] = -0.612 (valor base):
* La humedad a las 3pm se inclina mucho hacia predicciones de "no lloverá mañana"
* La presión a las 9am está asociada a menor probabilidad de lluvia

* Luego, la presión a las 3pm está asociada a una mayor probabilidad de lluvia, y en menor medida otras 66 features

La predicción final es f(x) = -1.862 (13.4% de probabilidad de lluvia el día posterior a la medición)

In [ ]:
shap.plots.bar(explanation)

* Las features que más empujan la predicción hacia "llueve" son Pressure3pm (+0.42), WindSpeed3pm (+0.24) y WindSpeed9am (+0.11).

* Las features que más empujan la predicción hacia "no lloverá" son Humidity3pm (-0.95), WindGustSpeed (-0.49) y Pressure9am (-0.33).

Las otras 61 features aportan poca información individualmente para resolver la incertidumbre sobre si mañana llueve o no, pero en conjunto empujan la predicción hacia "no lloverá" (-0.12)




Gráficas a nivel global

In [ ]:
explanation = shap.Explanation(values=shap_values, base_values=explainer.expected_value, feature_names=feature_names, data=X_test_scaled)

In [ ]:
shap.plots.bar(explanation)

El bar plot global nos indica el peso en promedio de cada feature en todo el dataset de prueba, pero no nos indica el sentido de cada feature, si empuja hacia "lloverá" o "no lloverá".

Vemos que las features con más peso del modelo son la humedad a la 3pm, presión a las 3pm, velocidad de ráfagas de viento, presión a las 9am y temperatura máxima del día donde se registró la medición, entre otras. Mientras que otras 61 features con poco peso individual, en conjunto aportan bastante información para la predicción.

In [ ]:
shap.plots.beeswarm(explanation)

A fin de interpretar el beeswarm plot, analizaremos las 5 primeras features:

Humedad a las 3pm: 
* poca humedad ambiente -> se asocia con menor probabilidad de lluvia
* mucha humedad ambiente -> se asocia con mayor probabilidad de lluvia

Presión atmosférica a las 3pm:
* baja presión atmosférica -> se asocia con mayor probabilidad de lluvia
* alta presión atmosférica -> se asocia con menor probabilidad de lluvia 

Velocidad de las ráfagas de viento:
* Ráfagas de pocos km/h -> se asocian a menor probabilidad de lluvia
* Ráfagas veloces -> se asocian a mayor probabilidad de lluvia

Presión atmosférica a las 9am:
* baja presión atmosférica matutina -> se asocia con menor probabilidad de lluvia
* alta presión atmosférica matutina -> se asocia con mayor probabilidad de lluvia 

Temperatura máxima del día: vemos que está muy concentrada en valores cercanos al 0, lo que indica que no aporta mucha información sobre si lloverá o no, pero en general, vemos un sesgo que indica:
* Una temperatura máxima baja está asociado a una mayor probabilidad de lluvia el día siguiente
* Una temperatura máxima alta está asociado a una menor probabilidad de lluvia el día siguiente